# 08 — Deploy Workshop 2 to your account (the runbook)

## The question

You have read the notebooks, the agents and MCP servers are written, the GraphQL
schema is defined, both React UIs are built — but none of it is *running in your
account* yet. A novice's natural question:

> "I have all this code. What is the exact, ordered set of commands that turns it into
> a working application I can open in a browser and log into — on a fresh account, from
> a SageMaker Studio kernel that has no Docker?"

This notebook is that runbook. Unlike the other notebooks (which teach a concept), this
one is **operational**: each cell is a step you actually run, in order, against your own
deployed stack. Every account-specific value — bucket names, the Cognito client id, the
hosted-UI domain — is read from your stack's **CloudFormation outputs**, never hardcoded.
That is the single most important habit this notebook teaches: *the deployed stack is the
source of truth; the runbook derives from it.*


## The concept

A Workshop 2 deployment has a strict order, and the order exists because of three
chicken-and-egg dependencies. Understanding *why* each step precedes the next matters
more than memorising the commands.

**1. The stack needs Workshop 1's outputs before it can synthesize.** The CDK app reads
the Neptune endpoints, the VPC, the private subnets, and the staging bucket from context.
Those values come from Workshop 1's `atlas-neptune-twotier` CloudFormation stack. The
pre-flight notebook (`00_preflight`) already wrote them into `cdk/cdk.json` for you — so
the bridge runs first.

**2. The runtimes need their dependencies in S3 before the stack deploys.** The 12
AgentCore runtimes import `bedrock_agentcore` and other packages that are not in the base
image. We package them as portable ZIPs and upload them to the staging bucket *ahead of*
`cdk deploy` (Option C — taught in `00_preflight`). CDK then references the S3 key. This
is what lets the whole deploy run **with no Docker** — the exact situation on a Studio
kernel. (The five Step Functions Lambdas are likewise Docker-free: four need only the
boto3 already in the Lambda runtime, and the one that needs `rdflib`/`pyshacl` installs
them locally because those wheels are pure-Python.)

**3. The OAuth callbacks need the CloudFront URLs, which only exist after the first
deploy.** Cognito's hosted-UI rejects any `redirect_uri` not pre-registered on the app
client. But your CloudFront distribution domains do not exist until the distributions are
created — by the deploy itself. That is the **two-pass callback flow**: deploy once to
create the distributions, read their URLs from the outputs, then redeploy registering
`<url>/callback` as the allowed callbacks.

Everything after that is application delivery: fill the UI environment from the outputs,
build the static export, sync it to the origin buckets, invalidate the CloudFront cache,
and log in. The order, end to end:

> bootstrap → bridge (WS1 outputs → cdk.json) → package runtimes → **deploy pass 1** →
> read UI URLs → **deploy pass 2** (callbacks) → fill `.env.local` → `next build` →
> `s3 sync` → invalidate → **log in + read the card**


> **Honesty note — what is proven vs. what this notebook teaches.** Every step below has
> been individually validated: the bucket names resolve from the live outputs, the UI
> `.env` keys match what the apps read, `next build` emits `out/` with the per-route
> `callback/index.html`, and `cdk synth` succeeds with **Docker stopped**. The *full
> clean-account end-to-end run* — WS0 → WS1 → WS2 from an empty account, in one sitting —
> is proven separately in the **F dry-run**, not here.
>
> One transitional note: the **first** `cdk deploy` that carries the Docker-free Lambda
> packaging change updates the five Step-Lambda code assets **once**; deploys after that
> show no Lambda changes.


## The build — the ordered runbook

Run these cells in order. The first reads your stack outputs into a Python dict that the
later cells reference, so **deploy the stack (steps 1–4) before relying on the
output-derived cells (5–8).** On the very first run the stack does not exist yet, so the
output read in the helper cell will be empty until pass-1 completes — that is expected.


### Setup — region + a helper that reads CloudFormation outputs

This is the one source of truth: every account-specific value below comes from here, so
nothing is hardcoded to one account.


In [ ]:
import subprocess, json, os

REGION = "us-east-1"            # WS2 is pinned to us-east-1 (see cdk/bin/atlas-workshop-2.ts)
STACK  = "AtlasWorkshop2"
USE_CASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))  # use-case-applications/

def stack_outputs(stack=STACK, region=REGION):
    """Return {OutputKey: OutputValue} for a deployed stack, or {} if it does not exist yet."""
    p = subprocess.run(
        ["aws", "cloudformation", "describe-stacks", "--stack-name", stack,
         "--region", region, "--query", "Stacks[0].Outputs", "--output", "json"],
        capture_output=True, text=True,
    )
    if p.returncode != 0:
        print(f"[outputs] stack {stack} not found yet (deploy steps 1-4 first).")
        return {}
    return {o["OutputKey"]: o["OutputValue"] for o in json.loads(p.stdout or "[]")}

out = stack_outputs()
print(f"{len(out)} outputs read from {STACK}." if out else "No outputs yet.")


### Step 1 — bootstrap your account for CDK (one time per account+region)

CDK needs a small "bootstrap" stack (an S3 assets bucket, a few roles) before it can
deploy anything. A fresh account has never been bootstrapped, so this is the true first
step. It is idempotent — safe to re-run.


In [ ]:
# One-time per account/region. Skip if `CDKToolkit` already exists in CloudFormation.
# !cd {USE_CASE_DIR}/cdk && npx cdk bootstrap aws://$(aws sts get-caller-identity --query Account --output text)/us-east-1
print("Run in a terminal (uncomment above to run from the notebook):")
print(f"  cd {USE_CASE_DIR}/cdk && npm install && npx cdk bootstrap")


### Step 2 — bridge + package the runtimes (Option C, Docker-free)

`00_preflight` already wrote Workshop 1's endpoints into `cdk/cdk.json` (the bridge). It
also explained Option C: package the 12 runtimes' dependencies into portable ZIPs and
upload them to the staging bucket, so `cdk deploy` needs **no Docker**. Run the build
script now (it prints the exact deploy command for the next step).


In [ ]:
# Packages all 12 runtimes -> uploads ZIPs to the WS1 staging bucket (no Docker).
# The script resolves the bucket from cdk/cdk.json (the preflight bridge populated it).
print("Run in a terminal:")
print(f"  cd {USE_CASE_DIR} && PY=python3 ./scripts/build-runtimes.sh build")
print()
print("Why no Docker: runtimes source their deps from S3 (fromS3); the 5 step Lambdas")
print("use the in-runtime boto3 + a local pure-Python install. `cdk synth` is Docker-free.")


### Step 3 — deploy pass 1 (creates everything, including the CloudFront distributions)

**Always pass `-c runtimeArtifactsS3Prefix=runtimes`** — it tells CDK to source the
runtimes from the ZIPs you just uploaded. *Omitting it ships runtimes with no
dependencies that provision green and then crash on first invocation.* (This is the flag
the older `spec/07` example used to omit — now reconciled.)


In [ ]:
print("Run in a terminal (pass 1 — no callbacks yet):")
print(f"  cd {USE_CASE_DIR}/cdk && npx cdk deploy -c runtimeArtifactsS3Prefix=runtimes")
print()
print("First deploy carrying the Docker-free Lambda packaging updates the 5 step-Lambda")
print("code assets ONCE; subsequent deploys show no Lambda changes.")


### Step 4 — deploy pass 2 (register the real CloudFront callbacks)

Now that pass 1 created the distributions, read their URLs from the outputs and redeploy
registering `<url>/callback`. This closes the chicken-and-egg: the callbacks could not be
known until the distributions existed.


In [ ]:
out = stack_outputs()   # refresh after pass 1
wholesale = out.get("WholesaleUiUrl", "<deploy pass 1 first>")
wealth    = out.get("WealthUiUrl",    "<deploy pass 1 first>")
callbacks = f"{wholesale}/callback,{wealth}/callback,http://localhost:3000/callback"
print("Wholesale UI:", wholesale)
print("Wealth UI:   ", wealth)
print()
print("Run in a terminal (pass 2 — registers the real callbacks):")
print(f"  cd {USE_CASE_DIR}/cdk && npx cdk deploy \\")
print(f"    -c runtimeArtifactsS3Prefix=runtimes \\")
print(f"    -c uiCallbackUrls={callbacks}")


### Step 5 — fill each UI's `.env.local` from the outputs

The apps read exactly three `NEXT_PUBLIC_*` variables, and Next.js `output: "export"`
**inlines them at build time** — so they must be set before `next build`. Each maps to a
stack output (one source of truth):

| Env var | Stack output |
|---|---|
| `NEXT_PUBLIC_APPSYNC_ENDPOINT` | `AppSyncEndpoint` |
| `NEXT_PUBLIC_COGNITO_CLIENT_ID` | `CognitoUserPoolWebClientId` |
| `NEXT_PUBLIC_COGNITO_DOMAIN` | `CognitoHostedUiDomain` |


In [ ]:
out = stack_outputs()
env_body = (
    f"NEXT_PUBLIC_APPSYNC_ENDPOINT={out.get('AppSyncEndpoint','')}\n"
    f"NEXT_PUBLIC_COGNITO_CLIENT_ID={out.get('CognitoUserPoolWebClientId','')}\n"
    f"NEXT_PUBLIC_COGNITO_DOMAIN={out.get('CognitoHostedUiDomain','')}\n"
)
for app in ("wholesale-ui", "wealth-ui"):
    path = os.path.join(USE_CASE_DIR, "apps", app, ".env.local")
    with open(path, "w") as f:
        f.write(env_body)
    print(f"wrote {path}")
print("\n.env.local (do NOT commit — gitignored; holds your account values):")
print(env_body)


### Step 6 — build each UI (static export → `out/`)

`output: "export"` produces a fully static site under `out/`, with each route as its own
`<route>/index.html` (this is why CloudFront needs the SpaRewrite function — see
`06_wholesale_ui`). Build *after* writing `.env.local` so the values are inlined.


In [ ]:
print("Run in a terminal (rebuild whenever .env.local changes):")
for app in ("wholesale-ui", "wealth-ui"):
    print(f"  cd {USE_CASE_DIR}/apps/{app} && npm install && npx next build   # emits out/")


### Step 7 — sync each `out/` to its CloudFront origin bucket

The origin buckets have CDK-generated names — there is no way to guess them, so we read
them from the outputs (`WholesaleBucketName`, `WealthBucketName`). This is the step the
older docs left undocumented: the deploy created *empty* buckets; this is what fills them.


In [ ]:
out = stack_outputs()
pairs = [("wholesale-ui", out.get("WholesaleBucketName","")),
         ("wealth-ui",    out.get("WealthBucketName",""))]
print("Run in a terminal:")
for app, bucket in pairs:
    if not bucket:
        print(f"  # {app}: bucket name not in outputs yet — deploy first")
        continue
    print(f"  aws s3 sync {USE_CASE_DIR}/apps/{app}/out/ s3://{bucket}/ --delete --region {REGION}")


### Step 8 — invalidate the CloudFront cache

S3 now has the new files, but CloudFront may still serve cached old ones. Invalidate `/*`
on each distribution. We resolve the distribution id from its domain (the `*UiUrl`
output) so nothing is hardcoded.


In [ ]:
out = stack_outputs()
def dist_id_for(domain_url):
    host = domain_url.replace("https://","").rstrip("/")
    p = subprocess.run(
        ["aws","cloudfront","list-distributions",
         "--query", f"DistributionList.Items[?DomainName=='{host}'].Id | [0]",
         "--output","text"], capture_output=True, text=True)
    return p.stdout.strip()

print("Run in a terminal:")
for key in ("WholesaleUiUrl","WealthUiUrl"):
    url = out.get(key,"")
    did = dist_id_for(url) if url else ""
    if did and did != "None":
        print(f"  aws cloudfront create-invalidation --distribution-id {did} --paths '/*'")
    else:
        print(f"  # {key}: distribution not found yet — deploy first")


## The verification — log in and read the card (G7)

The final proof is a human click-through (the runner's step). The cell below prints the
URLs and the exact success criteria. Open the Wholesale UI, sign in through the Cognito
hosted UI, and confirm the referral card renders the **two wealth signals** (the
`LargeDepositPattern` + `NoAdvisorCoverageSignal` you built in `05a_wealth_signals_build`).


In [ ]:
out = stack_outputs()
print("Open in a browser and sign in:")
print("  Wholesale UI:", out.get("WholesaleUiUrl","<not deployed>"))
print("  Wealth UI:   ", out.get("WealthUiUrl","<not deployed>"))
print()
print("Success criteria:")
print("  1. The hosted-UI login page loads (Cognito domain reachable).")
print("  2. After sign-in you are redirected to /callback and land in the app")
print("     (the CloudFront SpaRewrite function serves /callback/index.html).")
print("  3. The referral card shows BOTH wealth signals with provenance — proving the")
print("     full path UI -> AppSync -> MCP -> Neptune works end to end.")


In [ ]:
# Optional automated smoke check (no login): the surfaces should be reachable.
import urllib.request
def http_status(url):
    try:
        req = urllib.request.Request(url, method="GET")
        with urllib.request.urlopen(req, timeout=10) as r:
            return r.status
    except urllib.error.HTTPError as e:
        return e.code
    except Exception as e:
        return f"err: {e}"

out = stack_outputs()
for key in ("WholesaleUiUrl","WealthUiUrl"):
    url = out.get(key)
    if url:
        print(f"{key}: {http_status(url)}  ({url})")
        print(f"  /callback: {http_status(url + '/callback')}  (expect 200 via SpaRewrite)")


## What just changed

You now have a **running Workshop 2** in your own account: the GraphQL API, the 12
AgentCore runtimes (with their dependencies, sourced from S3 — no Docker was ever
required), the Step Functions referral orchestrator, Cognito with the OAuth code flow,
and both React UIs served from CloudFront with working login.

More importantly, you have the *habit* the runbook teaches: the deployed stack's
CloudFormation outputs are the source of truth, and every operational value — bucket
names, the Cognito client id, the hosted-UI domain — flows from them. Nothing is pinned
to one account, which is exactly what makes this workshop reproducible on a fresh one.

What becomes possible next: Phase 2 (the advisor experience, AgentCore Memory, the JWT
auth deep-dive) deploys onto this same stack — no new infrastructure, just the Wealth UI
and the Phase 2 notebooks you have already met.
